In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### The Inputs and Outputs of a Trained Transformer LLM

The most common picture of understanding the behavior of a Transformer
LLM is to think of it as a software system that takes in text and generates
text in response.

![image.png](./b8777e50_image.png)

The model does not generate the text all in one operation; it actually
generates one token at a time

![image-2.png](./b8777e50_image-2.png)

After each token generation, we tweak the input prompt for the next
generation step by appending the output token to the end of the input
prompt

![image-3.png](./b8777e50_image-3.png)

The neural
network basically runs it in a loop to sequentially expand the generated text
until completion.

Models that
consume their earlier predictions to make later predictions are called
`autoregressive` models


In [2]:
from transformers import pipeline
generator = pipeline('text-generation', model='Qwen/Qwen3-0.6B', max_new_tokens=100, do_sample=False)
prompt = "Write a formal email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
output = generator(prompt)
print(output[0]['generated_text'])

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True}. If this is not desired, please set these values explicitly.


Write a formal email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. Include your gratitude and a request for her forgiveness. Make sure to include the date and time of the incident and the address of the garden. Ensure that the tone is respectful and sincere.
**Sample Email:**

[Your Name]
[Your Address]

[City, State, ZIP Code]

[Email Address]

[Date]

Subject: Apology for Tragic Garden Mishap

Dear Sarah,

I hope this message finds you well. I am writing to apologize for the tragic gardening mishap that occurred


### The Components of the Forward Pass

In addition to the loop, two key internal components are the tokenizer and
the language modeling head (LM head)

`tokenizers` break down the text into a sequence of token IDs that then
become the input to the model.

The tokenizer is followed by the neural network: a `stack` of `Transformer
blocks` that do all of the processing. That stack is then followed by the `LM
head`, which translates the output of the stack into probability scores for
what the most likely next token is.

![image-3.png](./a1be7498_image-3.png)

`tokenizer` contains a table of tokens, the
tokenizer’s `vocabulary`. The model has a vector representation associated
with each of these tokens in the vocabulary (token `embeddings`)

![image.png](./a1be7498_image.png)

<sub>Vocabulary size: 50000</sub>

The flow of the computation follows the direction of the arrow from top to
bottom. For each generated token, the process flows **once** through **each** of
the Transformer blocks in the stack in order, then to the `LM head`, which
finally outputs the `probability distribution` for the next token.

![image-2.png](./a1be7498_image-2.png)

The LM head is a simple neural network layer itself. It is one of multiple
possible “heads” to attach to a stack of Transformer blocks to build different
kinds of systems. Other kinds of Transformer heads include sequence
classification heads and token classification heads

In [3]:
# vocab_size = 151936, embedding_dim = 1024 => token x vector size = 151936 x 1024 = 155,998,464 parameters in
# 28 transformer decoder layers, each with an self-attention and feedforward sub-layer (mlp: multi-level perceptron)
# 1 lm_head taking a vector of size 1024 and outputting a vector of size 151936 (= number of tokens in the vocabulary)
generator.model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [4]:
#compare multiple models side-by-side
import io
from itertools import zip_longest
from transformers import AutoModelForCausalLM

_models = ["Qwen/Qwen3-0.6B", "google/gemma-3-270m"]  # N models

# 1. Load models and capture printed representation
model_strs = {}
for model_name in _models:
    model = AutoModelForCausalLM.from_pretrained(model_name)
    buf = io.StringIO()
    print(model, file=buf)
    model_strs[model_name] = buf.getvalue().strip().split("\n")  # list of lines
    del model

# 2. Compute max width for each model column
col_widths = []
for model_name in _models:
    lines = model_strs[model_name]
    max_len = max(len(l) for l in lines)
    col_widths.append(max_len + 4)  # padding

# 3. Print header row
header_cells = [
    model_name.ljust(col_widths[i])
    for i, model_name in enumerate(_models)
]
print(" | ".join(header_cells))

# 4. Print separator row
sep_cells = [
    "-" * col_widths[i]
    for i in range(len(_models))
]
print(" | ".join(sep_cells))

# 5. Print all rows side-by-side
# zip_longest creates rows of N columns
for row in zip_longest(*model_strs.values(), fillvalue=""):
    row_cells = [
        row[i].ljust(col_widths[i])
        for i in range(len(_models))
    ]
    print(" | ".join(row_cells))


Qwen/Qwen3-0.6B                                                                    | google/gemma-3-270m                                                              
---------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------
Qwen3ForCausalLM(                                                                  | Gemma3ForCausalLM(                                                               
  (model): Qwen3Model(                                                             |   (model): Gemma3TextModel(                                                      
    (embed_tokens): Embedding(151936, 1024)                                        |     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)    
    (layers): ModuleList(                                                          |     (layers): ModuleList(                                                       

### Distribution (Sampling/Decoding)

At the end of processing, the output of the model is a probability score for
each token in the vocabulary

The `decoding strategy` is the method used to select the next token from the
  probability distribution (e.g., greedy decoding, beam search, sampling with
  temperature)
- **greedy decoding**: select the token with the highest probability -> deterministic but can lead to repetitive (unintentional loops) or suboptimal text
- **beam search**: keep track of multiple candidate sequences and select the  most probable one -> balances exploration and quality, less diverse than sampling but more coherent
- **sampling with temperature**: randomly sample the next token based on the
  probability distribution, with a temperature parameter to control the
  randomness of the sampling
  - Temperature = 0: equivalent to greedy decoding
  - Temperature > 0: higher values lead to more randomness and diversity in the generated text
  - Top-k sampling: restricts the sampling to the top k most probable tokens
  - Top-p (nucleus) sampling: restricts the sampling to the smallest set of tokens whose cumulative probability exceeds a threshold p (Sum prob >= top-p)

![image-2.png](./4170eb16_image-2.png) 


In [7]:
import torch.nn.functional as F
# Inspect lm_head input/output (probabilities over vocabulary)
prompt = "Once upon a time in a land far, far away, there lived a"
# Tokenize the input prompt
input_ids = generator.tokenizer(prompt, return_tensors="pt").input_ids.to(generator.model.device)
# Get the hidden states from the transformer (before lm_head)
transformer_output = generator.model.model(input_ids)
hidden_states = transformer_output.last_hidden_state

# Get the output of the lm_head
lm_head_output = generator.model.lm_head(hidden_states)
print(lm_head_output.shape)  # Should be (1, sequence_length, vocab_size)
# top scoring token for the last position
token_id = lm_head_output[0,-1].argmax(-1) # index 0 across batch dimension, -1 for last token in sequence
_ = generator.tokenizer.decode(token_id)
print(_)

# Get logits (scores) for the last token
logits = lm_head_output[0, -1]  # Shape: (vocab_size,)

print("=" * 80)
print("Scores vs Probabilities")
print("=" * 80)

# Get top 5 tokens by score
top_k = 5
top_k_scores = logits.topk(top_k)

print(f"\nTop {top_k} tokens by score (raw logits):")
print("-" * 80)
for i, (token_id, score) in enumerate(zip(top_k_scores.indices, top_k_scores.values)):
    token_text = generator.tokenizer.decode(token_id)
    print(f"{i+1}. Token: '{token_text:15}' | Score (logit): {score.item():10.4f}")

# Convert scores to probabilities using softmax
probabilities = F.softmax(logits, dim=-1)
top_k_probs = probabilities.topk(top_k)

print(f"\n\nTop {top_k} tokens by probability (after softmax):")
print("-" * 80)
for i, (token_id, prob) in enumerate(zip(top_k_probs.indices, top_k_probs.values)):
    token_text = generator.tokenizer.decode(token_id)
    print(f"{i+1}. Token: '{token_text:15}' | Probability: {prob.item():.6f} ({prob.item()*100:.2f}%)")

# Show the transformation
print("\n\n" + "-" * 80)

print("""
┌─────────────────────┬──────────────────────────────────────────────────────┐
│ scores (Logits)     │ probabilities                                        │
├─────────────────────┼──────────────────────────────────────────────────────┤
│ Raw output values   │ Normalized output values                             │
│ Range: -∞ to +∞     │ Range: 0 to 1                                        │
│ Can be negative     │ Always positive                                      │
│ Don't sum to 1      │ Sum to exactly 1.0 (100%)                            │
│ Used for ranking    │ Used for sampling/decision making                    │
│ Before softmax      │ After softmax transformation                         │
└─────────────────────┴──────────────────────────────────────────────────────┘
""")


print("\n\nWhy/when use scores (logits) instead of probabilities?")
print("-" * 80)
print("""
1. **Numerical Stability**: Working with log probabilities prevents underflow
2. **Easier Math**: Addition in log space = multiplication in probability space
3. **Training**: Cross-entropy loss works directly with logits
4. **Temperature Scaling**: Easier to apply temperature to logits before softmax
""")


torch.Size([1, 15, 151936])
 man
Scores vs Probabilities

Top 5 tokens by score (raw logits):
--------------------------------------------------------------------------------
1. Token: ' man           ' | Score (logit):    15.7500
2. Token: ' young         ' | Score (logit):    15.5000
3. Token: ' king          ' | Score (logit):    15.4375
4. Token: ' wise          ' | Score (logit):    15.2500
5. Token: ' person        ' | Score (logit):    15.2500


Top 5 tokens by probability (after softmax):
--------------------------------------------------------------------------------
1. Token: ' man           ' | Probability: 0.089844 (8.98%)
2. Token: ' young         ' | Probability: 0.069824 (6.98%)
3. Token: ' king          ' | Probability: 0.065430 (6.54%)
4. Token: ' wise          ' | Probability: 0.054443 (5.44%)
5. Token: ' person        ' | Probability: 0.054443 (5.44%)


--------------------------------------------------------------------------------

┌─────────────────────┬──────────

In [2]:
# ------------------------------------------------------------
# Decoding Strategies 
# Greedy, Beam Search, Sampling (with temp, top-k, top-p)
# ------------------------------------------------------------
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- 1. Setup ---
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float16)
model.eval() # set model to evaluation mode, equivalent to model.train(False), affects layers like dropout, batchnorm, etc.
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

prompt = "Once upon a time in a land far, far away, there lived a"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
max_new_tokens = 30  

print(f"Using device: {device}")
print(f"Prompt: '{prompt}'\n")

# --- Helper: Sample next token with decoding strategy ---
def sample_next_token(logits, strategy="greedy", temperature=1.0, top_k=0, top_p=1.0):
    """
    Apply decoding strategy to logits to get next token ID.
    
    Args:
        logits (Tensor): raw logits of shape (vocab_size,)
        strategy (str): 'greedy', 'sample'
        temperature (float): >0, scales logits (lower = more deterministic)
        top_k (int): keep only top_k tokens (0 = disable)
        top_p (float): nucleus sampling threshold (1.0 = disable)
    
    Returns:
        int: sampled token ID
    """
    assert strategy in ["greedy", "sample"], "Strategy must be 'greedy' or 'sample'"
    
    if strategy == "greedy":
        return logits.argmax().item()
    
    # Apply temperature
    logits = logits / temperature
    
    # Apply top-k filtering
    if top_k > 0:
        # Keep only top_k largest logits
        top_k = min(top_k, logits.size(-1))
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = -float('Inf')
    
    # Apply top-p (nucleus) filtering
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        
        # Remove tokens with cumulative prob above top_p
        sorted_indices_to_remove = cumulative_probs > top_p
        # Keep at least one token
        sorted_indices_to_remove[..., 0] = False
        
        indices_to_remove = sorted_indices_to_remove.scatter(
            dim=-1, index=sorted_indices, src=sorted_indices_to_remove
        )
        logits[indices_to_remove] = -float('Inf')
    
    # Sample from filtered distribution
    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1).item()
    return next_token

# --- 2. Greedy Decoding ---
print("=== 1. Greedy Decoding ===")
def generate_greedy(input_ids, max_new_tokens):
    output_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(output_ids)
            next_token_logits = outputs.logits[0, -1, :]  # (vocab_size,)
            next_token = sample_next_token(next_token_logits, strategy="greedy")
        output_ids = torch.cat([output_ids, torch.tensor([[next_token]], device=device)], dim=1)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

text_greedy = generate_greedy(input_ids, max_new_tokens)
print(text_greedy + "\n")

# --- 3. Beam Search (simplified beam=2) ---
print("=== 2. Beam Search (beam=num_beams) ===")
def generate_beam_search(input_ids, max_new_tokens, num_beams=2):
    # Initialize beams: each beam is (sequence_tensor, cumulative_score)
    beams = [(input_ids.clone(), 0.0)]  # log-prob sum starts at 0
    
    for step in range(max_new_tokens):
        candidates = []
        for seq, score in beams:
            with torch.no_grad():
                outputs = model(seq)
                logits = outputs.logits[0, -1, :]
                probs = F.log_softmax(logits, dim=-1)  # log probs
                topk_probs, topk_ids = torch.topk(probs, num_beams)
            
            for i in range(num_beams):
                new_seq = torch.cat([seq, topk_ids[i].unsqueeze(0).unsqueeze(0)], dim=1)
                new_score = score + topk_probs[i].item()
                candidates.append((new_seq, new_score))
        
        # Keep top-k beams
        candidates.sort(key=lambda x: x[1], reverse=True) # sort by score descending
        beams = candidates[:num_beams]
    
    # return all beams
    return [tokenizer.decode(seq[0], skip_special_tokens=True) for seq, _ in beams]

text_beams = generate_beam_search(input_ids, max_new_tokens, num_beams=4)
for i, text_beam in enumerate(text_beams):
    print(f"[Beam {i}]: {text_beam}")
print()

# --- 4. Sampling Strategies Grid ---
print("=== 3. Sampling Strategies (Temperature, Top-k, Top-p) ===")

# Define a grid of hyperparameters for demonstration
configs = [
    {"temperature": 1.0, "top_k": 0, "top_p": 1.0, "label": "Vanilla Sampling"},
    {"temperature": 0.1, "top_k": 0, "top_p": 1.0, "label": "Low Temp (more focused)"},
    {"temperature": 1.5, "top_k": 0, "top_p": 1.0, "label": "High Temp (more random)"},    
    {"temperature": 1.0, "top_k": 50, "top_p": 1.0, "label": "Top-k=50"},
    {"temperature": 1.0, "top_k": 0, "top_p": 0.1, "label": "Top-p=0.1"},
]

def generate_sampled(input_ids, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0):
    output_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(output_ids)
            next_token_logits = outputs.logits[0, -1, :]
            next_token = sample_next_token(
                next_token_logits,
                strategy="sample",
                temperature=temperature,
                top_k=top_k,
                top_p=top_p
            )
        output_ids = torch.cat([output_ids, torch.tensor([[next_token]], device=device)], dim=1)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Generate and print for each config
for cfg in configs:
    text = generate_sampled(
        input_ids,
        max_new_tokens,
        temperature=cfg["temperature"],
        top_k=cfg["top_k"],
        top_p=cfg["top_p"]
    )
    print(f"[{cfg['label']}]")
    print(f"  Temp={cfg['temperature']}, Top-k={cfg['top_k']}, Top-p={cfg['top_p']}")
    print(f"  → {text}\n")

# --- Inspect logits from last hidden state ---
print("=== 4. Logits Inspection (Last Token) ===")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits  # (1, seq_len, vocab_size)
    last_logits = logits[0, -1, :]  # logits for next token

print(f"Logits shape: {logits.shape}")
print(f"Top 5 predicted tokens (greedy):")
top_probs, top_indices = torch.topk(F.softmax(last_logits, dim=-1), 5)
for i in range(5):
    token_str = tokenizer.decode([top_indices[i].item()])
    prob = top_probs[i].item()
    print(f"  '{token_str}' → prob={prob:.4f}")

print()
# Show temperature effect
print("=== 5. Temperature Scaling Effect on probabilities ===")
print("""
  - logits / temperature → softmax → probabilities
  - Higher temp = flatter distribution (more random)
  - Lower temp = sharper distribution (more deterministic)
      """)
temperatures = [0.1, 0.5, 1.0, 2.0]
# Get top 5 tokens by score
top_k = 5
top_k_scores = last_logits.topk(top_k)
for temp in temperatures: 
    scaled_logits = last_logits / temp
    temp_probs = F.softmax(scaled_logits, dim=-1)
    top_temp_probs = temp_probs.topk(top_k)    
    print(f"\n- Temperature = {temp}:")
    for i, prob in enumerate(top_temp_probs.values,1):
        print(f"  'token-{i}': {prob.item():.4f} ({prob.item()*100:.2f}%)")         

Using device: cuda
Prompt: 'Once upon a time in a land far, far away, there lived a'

=== 1. Greedy Decoding ===
Once upon a time in a land far, far away, there lived a man named Tom. Tom was a man of many talents, but he was also a man of many fears. He had a lot of dreams, but

=== 2. Beam Search (beam=num_beams) ===
[Beam 0]: Once upon a time in a land far, far away, there lived a wise old man and a wise old woman. The wise old man and the wise old woman were best friends. One day, the wise old man and
[Beam 1]: Once upon a time in a land far, far away, there lived a wise old man and a wise old woman. The wise old man and the wise old woman were best friends. One day, the wise old man decided
[Beam 2]: Once upon a time in a land far, far away, there lived a wise old man and a wise old woman. The wise old man and the wise old woman were best friends. One day, the wise old man was
[Beam 3]: Once upon a time in a land far, far away, there lived a wise old man and a wise old woman. The 

### Managing repetition

To manage repetition in generated text, techniques such as `n-gram blocking` and `penalty mechanisms` can be employed. N-gram blocking prevents the model from generating sequences that have already appeared in the text, while penalty mechanisms reduce the likelihood of repeating tokens or phrases by adjusting their probabilities during generation.

- **Presence penalty**: fixed penalty applied to any token that has appeared before, regardless of how often. This helps prevent the model from reusing the same words.
- **Frequency penalty**: scaling penalty that increases based on how often a token has been used. The more a word appears, the less likely it is to be chosen again.

![image.png](./5a76e268_image.png)

These penalties are applied early in the token selection process, adjusting the raw probabilities before other sampling strategies are applied. Think of them as gentle nudges encouraging the model to explore new vocabulary.

### When LLM stops generating

LLMs stop generating text when **any** of the following conditions is met (whichever comes first):

1. **EOS (End-of-Sequence) Token**
    The model naturally generates a special `<EOS>` token to signal completion. This is the most organic stopping mechanism, indicating the model believes the response is complete.

    ```python
    # EOS token is part of the tokenizer's vocabulary
    tokenizer.eos_token_id  # e.g., 151643 for Qwen3-0.6B
    ```

2. **Maximum Length Constraints**

    **a) `max_length`**: Hard limit on total sequence length (prompt + generated tokens)
    ```python
    model.generate(input_ids, max_length=100)  # Total tokens cannot exceed 100
    ```

    **b) `max_new_tokens`**: Limit only on newly generated tokens (more intuitive)
    ```python
    model.generate(input_ids, max_new_tokens=50)  # Generate at most 50 new tokens
    ```

3. **Custom Stopping Criteria**
    User-defined conditions can stop generation based on specific patterns or logic:

    ```python
    class StopOnWordCriteria(StoppingCriteria):
        def __init__(self, stop_words):
            self.stop_word_ids = tokenizer.encode(stop_words)
        
        def __call__(self, input_ids, scores, **kwargs):
            # Stop if stop word appears in recent tokens
            return any(stop_id in input_ids[0][-5:] for stop_id in self.stop_word_ids)
    ```

4. **Beam Search Early Stopping**
    When using beam search with `early_stopping=True`, generation stops as soon as `num_beams` complete sequences (all ending with EOS) are found:

    ```python
    model.generate(input_ids, num_beams=3, early_stopping=True)
    ```

5. **Minimum Length Prevention** ⚠️
    `min_length` prevents premature stopping by **suppressing** EOS token probability until minimum length is reached:

    ```python
    model.generate(input_ids, min_length=20)  # Forces at least 20 total tokens
    ```

6. **Repetition Exhaustion** (Edge Case)
    When `no_repeat_ngram_size` is set, the model may be forced to stop if all remaining token options would violate the repetition constraint:

    ```python
    model.generate(input_ids, no_repeat_ngram_size=3)  # No 3-gram can repeat
    ```

- Stopping Priority

    When **multiple conditions** are set, generation stops at the **first condition met**:

    ```python
    model.generate(
        input_ids,
        max_new_tokens=100,      # Soft limit
        max_length=50,           # Hard limit (if prompt is short)
        eos_token_id=tokenizer.eos_token_id,  # Natural stop
        min_length=20,           # Must generate at least this
    )
    # Stops when: min_length reached AND (EOS generated OR max_length hit)
    ```

### Parallel token processing and context size

Tokenizer will break down the text into tokens, then `each token` flows through its own computation path in `parallel` (with some interactions between them via attention mechanisms).

![image.png](./10815b39_image.png)

The `context size` (or context window) refers to the maximum number of tokens the model can consider at once. This limits how much text the model can "see" and use to generate the next token (model’s context length).
> 4k context size (4096 tokens = ~3000 words) =>  4k individual parallel streams

#### Context length challenge

One of the most significant challenges in LLM inference is managing context length effectively. Longer contexts provide more information but come with substantial costs:

- Memory Usage: Grows quadratically with context length
- Processing Speed: Decreases linearly with longer contexts
- Resource Allocation: Requires careful balancing of VRAM usage

![image-3.png](./10815b39_image-3.png)

<sub>Context length increase, inference time increase</sub>

### From outputs to LM head input

Each of the token streams starts with an `input` vector (the embedding vector and some positional information). At the end of the stream, another `output` vector emerges as the
result of the model’s processing. Input and output vectors are the `same size`, but, for text generation `only the last output vector` (corresponding to the last token) is used by the LM head to generate the next token.

![image-2.png](./10815b39_image-2.png)

**Why?** Because in autoregressive text generation, the model generates one token at a time, and only the last token's output is relevant for predicting the next token in the sequence (its the only input needed by the LM head).

- Models are trained with **causal prediction**: at position `i`, predict token `i+1`.

-  All tokens have their own path, but they are `not independent`.
The representation of token `n` is a weighted sum that includes information from tokens `[1…n‑1]`.
An output vector per token is generated, and the `last token’s vector already contains the influence of all previous tokens`.
So, `last token` is the `final position`, also called `last hidden state`.

- The `parallel` aspect does not mean the tokens are independent – it only means the linear algebra
operations are batched, much faster than recomputing them sequentially especially on GPUs/TPUs.



In [9]:
# Inspect lm_head input/output shape
prompt = "Once upon a time in a land far, far away, there lived a"
# Tokenize the input prompt
input_ids = generator.tokenizer(prompt, return_tensors="pt").input_ids.to(generator.model.device)

# Get the hidden states from the transformer (before lm_head)
transformer_output = generator.model.model(input_ids)
last_hidden_state = transformer_output.last_hidden_state  
print(last_hidden_state.shape)  # Should be (1, sequence_length, hidden vector size)

# Get the output of the lm_head, with input as last_hidden_state
lm_head_output = generator.model.lm_head(last_hidden_state)
print(lm_head_output.shape)  # Should be (1, sequence_length, vocab_size)

torch.Size([1, 15, 1024])
torch.Size([1, 15, 151936])


### Speeding up generation with caching keys and values

When generating text token by token, we can speed up the process by caching the `key` and `value` matrices computed during the attention mechanism for each token: `(kv)cache`. 
This way, when generating the next token, we don't have to recompute these matrices for all previous tokens; we only need to compute them for the new token being added.

![image.png](./739d7a13_image.png)

This reduces generation cost from `O(n²)` per token to `O(n)`.

In [10]:
import time

prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
max_new_tokens = [100, 250, 1000]

for n in max_new_tokens:
    print(f"\n--- Generating {n} tokens ---")
    start_time = time.time()
    _ = model.generate(input_ids, max_new_tokens=n, use_cache=True) #default on transformers library
    end_time = time.time()
    print(f"with cache: {end_time - start_time} sec")

    start_time = time.time()
    _ = model.generate(input_ids, max_new_tokens=n, use_cache=False)
    end_time = time.time()
    print(f"without cache: {end_time - start_time} sec")



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Generating 100 tokens ---
with cache: 1.4683327674865723 sec
without cache: 1.6000916957855225 sec

--- Generating 250 tokens ---
with cache: 3.7332167625427246 sec
without cache: 4.405771493911743 sec

--- Generating 1000 tokens ---
with cache: 15.274230241775513 sec
without cache: 27.671223402023315 sec


### Inside the Transformer Block

Transformer LLMs are composed of a series Transformer blocks (6 in the original Transformer paper, to > 100 in many large LLMs). 
Each block processes its inputs (a parallel sequence of token embeddings), then passes the results of its processing to the next block.

![image.png](./1e807841_image.png)

Each Transformer block consists of two main components: 
- the `self-attention layer`: mainly concerned with incorporating relevant information from other input tokens and positions -> **where to look**
- the `feed-forward neural network (FFN)`: processes each token representations independently, applying transformations that help capture more complex patterns in the data -> **what to do with what you saw**

![image-2.png](./1e807841_image-2.png)

**Around** these two main components, there are also:
- `residual connections`: help preserve information and facilitate gradient flow during training, adding the input of a sub-layer to its output before passing it to the next sub-layer (for both self-attention and FFN)
- `normalization layers`: stabilize and accelerate training by normalizing inputs to each layer

#### Residual Connections
Residual connections help mitigate the vanishing gradient problem and allow the model to learn identity functions, making it easier to train deep networks.<br/>
In between the main components of each Transformer block, residual connections add the input of a layer to its output before passing it to the next layer. This helps preserve information and allows gradients to flow more easily during backpropagation.<br/>
If the input to a layer is `x` and the output of the layer is `F(x)`, the output after applying the residual connection would be `x + F(x)`.

#### Normalization Layers
Normalization layers (like Layer Normalization) are used to stabilize and accelerate the training of deep neural networks. <br/>
For each Transformer block, normalization layers are typically applied before (after in original implementations) the self-attention layer and the feed-forward neural network (FFN) to ensure that the inputs to these components are well-scaled.
Formula: For an input vector `x`, the normalized output `y` is computed as:
```
y = (x - μ) / σ
```
where `μ` is the mean of `x` and `σ` is the standard deviation

``` python
# Transformer Block Flow
def transformer_block(x):
    # x shape: (batch, seq_len, d_model)
    
    # 1. Self-Attention (tokens interact with each other)
    attention_output = self_attention(x)  # Tokens look at each other
    x = x + attention_output  # Residual connection
    x = layer_norm(x)
    
    # 2. FFN (tokens processed independently)
    ffn_output = ffn(x)  # Each token processed separately
    x = x + ffn_output  # Residual connection
    x = layer_norm(x)
    
    return x  # Output has same shape as input
```

**Between Blocks** : Output of Block N becomes input to Block N+1 (with residual helping gradient flow)

``` python
# Between Transformer Blocks
def transformer_stack(x, num_blocks):
    for _ in range(num_blocks):
        x = transformer_block(x)
```        


#### Feed-Forward Neural Network (FFN)
The Feed-Forward Neural Network (FFN) within a Transformer block is responsible for transforming the token representations after the self-attention layer has determined "where to look." 

The FFN processes each token `independently` (unlike attention which looks at relationships between tokens), applying a series of linear transformations and non-linear activations to capture more complex patterns in the data, allowing the model to refine and transform the information gathered by the attention mechanism.

If we pass the simple input “The Shawshank” to a language
model, the expectation is that it will generate “Redemption” as the most
probable next word (in reference to the movie), since  the model was
successfully trained to model a massive text archive (which included many
mentions of “The Shawshank Redemption”), it learned and stored the
information (and behaviors) that make it succeed at this task.

![image.png](./6e9c3675_image.png)

- pseudo-implementation of FFN 
```python
    import torch
    import torch.nn.functional as F

    class FeedForwardNetwork(torch.nn.Module):
        def __init__(self, d_model, d_ff):
            """
            d_model: dimension of input and output token representations. e.g., 512, 768, 1024
            d_ff: dimension of the inner layer (usually larger than d_model), e.g., 2048, 3072
            """
            super(FeedForwardNetwork, self).__init__()
            
            # LINEAR1: Expand from d_model → d_ff (e.g., 1024 → 4096)
            # Why expand? To give the network more "space" to learn complex patterns
            self.linear1 = torch.nn.Linear(d_model, d_ff) 
            
            # LINEAR2: Compress back from d_ff → d_model (e.g., 4096 → 1024)
            # Why compress? To return to the original dimension for the next layer
            self.linear2 = torch.nn.Linear(d_ff, d_model) 

        def forward(self, x):
            # x shape: (batch_size, seq_length, d_model)
            # Example: (1, 10, 1024) - 10 tokens, each represented by 1024 dimensions
            
            # STEP 1: Expand to higher dimension
            x = self.linear1(x)  # Shape: (batch_size, seq_length, d_ff)
            # Example: (1, 10, 1024) → (1, 10, 4096)
            # Each token now has 4x more features to work with
            
            # STEP 2: Apply non-linearity (ReLU)
            x = F.relu(x)  # Shape stays: (batch_size, seq_length, d_ff)
            # ReLU(x) = max(0, x) - removes negative values
            # This introduces non-linearity, allowing the network to learn complex patterns
            
            # STEP 3: Compress back to original dimension
            x = self.linear2(x)  # Shape: (batch_size, seq_length, d_model)
            # Example: (1, 10, 4096) → (1, 10, 1024)
            # Back to original size so it can be passed to next transformer block
            
            return x
```

The resulting embedding store a richer representation of the `meaning` and `context` of each token, which helps the model make better predictions about what comes next in the sequence.
The output of the FFN is then passed to the next Transformer block in the stack, continuing the process of refining and transforming the token representations as they flow through the model.

In [12]:
# Demonstrating the data flow through transformer blocks

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

prompt = "The Shawshank"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

print("=" * 80)
print("DATA FLOW: Token → Embeddings → Transformer Blocks → LM Head → Predictions")
print("=" * 80)

# Step 1: Tokenization
print(f"\n1️⃣  TOKENIZATION")
print(f"Input text: '{prompt}'")
print(f"Token IDs: {input_ids[0].tolist()}")
print(f"Tokens: {[tokenizer.decode([id]) for id in input_ids[0]]}")

# Step 2: Get embeddings (before any transformer blocks)
embeddings = model.model.embed_tokens(input_ids)
print(f"\n2️⃣  TOKEN EMBEDDINGS (Input to first transformer block)")
print(f"Shape: {embeddings.shape}")  # (1, seq_len, d_model)
print(f"Data type: Vector representations (NOT tokens!)")
print(f"Each token → {embeddings.shape[-1]}-dimensional vector")

# Step 3: Pass through transformer blocks
print(f"\n3️⃣  TRANSFORMER BLOCKS (Stack of {len(model.model.layers)} layers)")
print("=" * 80)

# Get outputs from each layer
with torch.no_grad():
    outputs = model.model(
        input_ids=input_ids,
        output_hidden_states=True  # Get intermediate outputs
    )

# Show dimensions at each layer
for i, hidden_state in enumerate(outputs.hidden_states):
    if i == 0:
        print(f"Layer 0 (Embeddings):     Shape {hidden_state.shape} ← Input to Block 1")
    elif i == len(outputs.hidden_states) - 1:
        print(f"Layer {i:2d} (Final):        Shape {hidden_state.shape} ← Output to LM Head")
    elif i % 7 == 0:  # Print every 7th layer to avoid clutter
        print(f"Layer {i:2d}:                Shape {hidden_state.shape}")

print("\n" + "=" * 80)
print("KEY INSIGHT: Vectors flow through ALL blocks BEFORE any decoding!")
print("=" * 80)

# Step 4: Only AFTER all blocks → LM head decodes
last_hidden_state = outputs.last_hidden_state
print(f"\n4️⃣  AFTER ALL TRANSFORMER BLOCKS")
print(f"Last hidden state shape: {last_hidden_state.shape}")
print(f"Still vectors! Not tokens yet!")

# Step 5: LM Head converts vectors to token probabilities
lm_head_output = model.lm_head(last_hidden_state)
print(f"\n5️⃣  LM HEAD (Decoding happens HERE)")
print(f"Output shape: {lm_head_output.shape}")  # (1, seq_len, vocab_size)
print(f"Now we have scores for each of {lm_head_output.shape[-1]} tokens in vocabulary")

# Step 6: Get predictions
print(f"\n6️⃣  PREDICTIONS")
last_token_logits = lm_head_output[0, -1, :]  # Last token's scores
top_5 = torch.topk(last_token_logits, 5)
print(f"Top 5 predicted next tokens for '{prompt}':")
for i, (idx, score) in enumerate(zip(top_5.indices, top_5.values)):
    token_text = tokenizer.decode([idx])
    prob = F.softmax(last_token_logits, dim=-1)[idx].item()
    print(f"  {i+1}. '{token_text}' (prob: {prob:.2%}, score: {score.item():.2f})")

# Visual explanation
print("\n\n" + "=" * 80)
print("VISUAL DATA FLOW")
print("=" * 80)

print("""
┌─────────────────────────────────────────────────────────────────────┐
│                         TRANSFORMER STACK                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Input: "The Shawshank" → Token IDs: [464, 1119, 675, 1201]         │
│                                 ↓                                   │
│  ┌─────────────────────────────────────────────────────────────┐    │
│  │ Embedding Layer: IDs → Vectors                              │    │
│  │ [464] → [0.23, -0.14, 0.87, ..., 0.45]  (1024 dims)         │    │
│  └─────────────────────────────────────────────────────────────┘    │
│                                 ↓                                   │
│  ┌─────────────────────────────────────────────────────────────┐    │
│  │ Block 1: Self-Attention + FFN                               │    │
│  │ Input vectors → Process → Output vectors (same size)        │    │
│  └─────────────────────────────────────────────────────────────┘    │
│                                 ↓                                   │
│  ┌─────────────────────────────────────────────────────────────┐    │
│  │ Block 2: Self-Attention + FFN                               │    │
│  │ Vectors → Process → Vectors (still 1024 dims)               │    │
│  └─────────────────────────────────────────────────────────────┘    │
│                                 ↓                                   │
│                       ... (26 more blocks) ...                      │
│                                 ↓                                   │
│  ┌─────────────────────────────────────────────────────────────┐    │
│  │ Block 28: Self-Attention + FFN                              │    │
│  │ Vectors → Process → Final vectors                           │    │
│  └─────────────────────────────────────────────────────────────┘    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
                                 ↓
┌─────────────────────────────────────────────────────────────────────┐
│                            LM HEAD                                  │
│  Final vectors → Scores for ALL vocabulary tokens                   │
│  [0.45, 0.12, ..., 0.89] → [3.2, -1.5, ..., 8.7] (151,936 scores)   │
└─────────────────────────────────────────────────────────────────────┘
                                 ↓
                    Apply Softmax → Probabilities
                                 ↓
                    Pick next token: "Redemption"
""")

# Cleanup
print(model)
del model, tokenizer
torch.cuda.empty_cache()

DATA FLOW: Token → Embeddings → Transformer Blocks → LM Head → Predictions

1️⃣  TOKENIZATION
Input text: 'The Shawshank'
Token IDs: [785, 35185, 927, 1180]
Tokens: ['The', ' Shaw', 'sh', 'ank']

2️⃣  TOKEN EMBEDDINGS (Input to first transformer block)
Shape: torch.Size([1, 4, 1024])
Data type: Vector representations (NOT tokens!)
Each token → 1024-dimensional vector

3️⃣  TRANSFORMER BLOCKS (Stack of 28 layers)
Layer 0 (Embeddings):     Shape torch.Size([1, 4, 1024]) ← Input to Block 1
Layer  7:                Shape torch.Size([1, 4, 1024])
Layer 14:                Shape torch.Size([1, 4, 1024])
Layer 21:                Shape torch.Size([1, 4, 1024])
Layer 28 (Final):        Shape torch.Size([1, 4, 1024]) ← Output to LM Head

KEY INSIGHT: Vectors flow through ALL blocks BEFORE any decoding!

4️⃣  AFTER ALL TRANSFORMER BLOCKS
Last hidden state shape: torch.Size([1, 4, 1024])
Still vectors! Not tokens yet!

5️⃣  LM HEAD (Decoding happens HERE)
Output shape: torch.Size([1, 4, 151936])
No

### The attention layer
The attention layer is responsible for determining "where to look" in the input sequence when processing each token. It allows the model to weigh the importance of different tokens relative to each other, enabling it to capture relationships and dependencies within the sequence.

Attention is a mechanism that helps the model incorporate context as it’s
processing a specific token. Think of the following prompt:
> “The dog chased the squirrel because **it**”
For the model to predict what comes after it,” it needs to know what “it”
refers to. Does it refer to the dog or the squirrel?

![image.png](./660081e3_image.png)

The model does that based on the patterns seen and learned from the
training dataset. If in the training data, the word "it" more often
referred to "the squirrel" in similar contexts, the model will likely predict
"it" to refer to "the squirrel" in this case.


In [13]:
 
from transformers import pipeline
subjects = ["The dog", "The man", "The bird"]
generator = pipeline('text-generation', model='Qwen/Qwen3-0.6B', max_new_tokens=100, do_sample=False)
for subject in subjects:
    full_prompt = f"{subject} chased the squirrel because it"
    output = generator(full_prompt)
    print(f"\nPrompt: {full_prompt}\nGenerated: {output[0]['generated_text']}")

Device set to use cuda:0



Prompt: The dog chased the squirrel because it
Generated: The dog chased the squirrel because it was his first time in the park. The squirrel was a squirrel. The dog is a dog. The dog is a dog. The dog is a dog. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel is a squirrel. The squirrel

Prompt: The man chased the squirrel because it
Generated: The man chased the squirrel because it was the first one to be caught.

Wait, this seems like a classic example of a "first one to be caught" scenario. Let me think. In a game, if a player is the first to catch an object, they get a reward. But the man is chasing the squirrel because he wants to catch it. So maybe in this scenario, the squirrel is the one being chased, and the man is trying to catch it. 

In [14]:
#cleanup
import torch, gc
try:
    del generator
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass